In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import webbrowser
import tempfile
import base64
from io import BytesIO

def fig_to_base64(fig):
    """ 
    Convert matplotlib figure to a png image encoded as base64 string for embedding in the HTML report. 
    Save the image in memory instead of writing it to disk.
    """
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return encoded

def plot_fruits_per_image(df):
    fpg = df.groupby('label')['fruit_id'].count().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    
    ax.bar(fpg.index, fpg.values, color='#4aaa88', edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Image', fontsize=11)
    ax.set_ylabel('Num fruits', fontsize=11)
    ax.set_title('Fruits per image', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    
    fig.tight_layout()
    return fig_to_base64(fig)

In [ ]:

def generate_report(csv_path):
    df = pd.read_csv(csv_path)

    # Variables
    total_fruits = len(df)
    total_labels = df['label'].nunique()
    total_images = df['image_name'].nunique()
    rows, cols = df.shape
    locules = df['n_locules'].value_counts().sort_index().reset_index()
    locules.columns = ['Num locules', 'Count']
    locules['%'] = (locules['Count'] / total_fruits * 100).round(1)

    overview_rows = f"""
      <tr><td>Total fruits analyzed</td><td>{total_fruits}</td></tr>
      <tr><td>Images</td><td>{total_images}</td></tr>
      <tr><td>Labels</td><td>{total_labels}</td></tr>
      <tr><td>Samples (rows)</td><td>{rows}</td></tr>
      <tr><td>Variables (cols)</td><td>{cols}</td></tr>
      <tr><td>Total observarions (cols*rows)</td><td>{rows*cols}</td></tr>
    """

    locules_rows = '\n'.join(
        f"  <tr><td>{r['Num locules']}</td><td>{r['Count']}</td><td>{r['%']}%</td></tr>"
        for _, r in locules.iterrows()
    )

    fpg_img = plot_fruits_per_image(df)

    # Build the report body with HTML
    html = f"""
    <!DOCTYPE html>
    <html>
    
    <head>
      <meta charset="UTF-8">
      <title>Morphology Report</title>
      <style>
        body {{ font-family: sans-serif; max-width: 960px; margin: 40px auto; padding: 0 20px; color: #222; }}
        h1 {{ border-bottom: 2px solid #ccc; padding-bottom: 8px; }}
        h2 {{ margin-top: 40px; border-bottom: 1px solid #eee; padding-bottom: 4px; }}
        img {{ max-width: 100%; margin-top: 16px; }}
        table {{ border-collapse: collapse; margin-top: 12px; }}
        th, td {{ border: 1px solid #ddd; padding: 8px 16px; text-align: left; font-size: 13px; }}
        th {{ background: #f5f5f5; font-weight: bold; }}
        tr:nth-child(even) {{ background: #fafafa; }}
      </style>
    </head>
    
    <body>
    
    <h1>Fruit Morphology Report</h1>
    
    <h2>Overview</h2>
    <table>
      <tr><th>Metric</th><th>Value</th></tr>
    {overview_rows}
    </table>
    
    <h2>Fruits per image</h2>
    <img src="data:image/png;base64,{fpg_img}" alt="Fruits per image">
    
    <h2>Locule distribution</h2>
    <table>
      <tr><th>Num locules</th><th>Fruits</th><th>%</th></tr>
    {locules_rows}
    </table>
    
    </body>
    </html>
    
    """
    tmp = tempfile.NamedTemporaryFile(suffix='.html', delete=False, mode='w', encoding='utf-8')
    tmp.write(html)
    tmp.close()
    webbrowser.open(f"file://{tmp.name}")
    print(f"Report URL: {tmp.name}")

# # Create report
# csv_path = "/Users/alejandra/Documents/Projects_2026/Traitly_Paper/Scanner/cranberry_2025/stamps/Images_from_PDF/Results/morphology_results.csv"
# generate_report(csv_path)

In [ ]:
import tracemalloc

tracemalloc.start()
generate_report(csv_path)
current, peak = tracemalloc.get_traced_memory()
print(f"actual memory: {current / 1024 / 1024:.2f} MB")
print(f"peak: {peak / 1024 / 1024:.2f} MB")
tracemalloc.stop()